In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import geopandas as gpd

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets/"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
BARPLOT_ALPHA = 0.7

## Datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_ca.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_ca.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_alaska.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_alaska.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

In [ ]:
# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'SJM', 'CENTRALASIA', 'ALA', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

### Model assets:

#### Pooled set:

In [ ]:
# Build glacier→region mapping from all source regions (no ranking needed)
df_all_glaciers = pd.concat(
    [
        res_xreg_by_source[region]["df_train"][[
            "GLACIER"
        ]].drop_duplicates().assign(region=region) for region in SOURCE_REGIONS
    ],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

# Add measurement counts (needed by build_assets_from_glacier_list internally? No — only used by build_glacier_subsets)
# df_ranked just needs columns: "glacier", "region"  ← that's all build_assets uses
glaciers_all = df_all_glaciers["glacier"].tolist()
print(f"Total unique glaciers across all source regions: {len(glaciers_all)}")

assets_full = build_assets_from_glacier_list(
    glaciers=glaciers_all,
    df_ranked=df_all_glaciers,  # acts as the gl→region lookup
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_path=str(CACHE_DIR / "assets_foundation_100pct.joblib"),
    force_recompute=False,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    src_regions=SOURCE_REGIONS,
)

# Failsafe: check that pooled training data only contains source region codes
actual_codes = set(df_all_glaciers.region.unique())
expected_codes = set(SOURCE_REGIONS)
unexpected = actual_codes - expected_codes
if unexpected:
    raise ValueError(
        f"Pooled training set contains unexpected SOURCE_CODEs: {unexpected}")
else:
    print(f"✓ Pooled set SOURCE_CODEs OK: {actual_codes}")

print(f"Pooled model assets: {len(assets_full['ds_train'])} sequences")

## Grid search:

In [ ]:
from itertools import product

def generate_transformer_param_sets(const_params, param_grid):
    """
    Generate all valid transformer parameter combinations.
    Filters out combinations where nhead does not divide d_model.
    """
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]

    param_sets = []
    for combo in product(*values):
        params = dict(zip(keys, combo))

        # nhead must divide d_model
        if params["d_model"] % params["nhead"] != 0:
            continue

        # dim_feedforward should be >= d_model (not strictly required but sensible)
        if params["dim_feedforward"] < params["d_model"]:
            continue

        # no static MLP -> ignore static_hidden & static_dropout
        if params["static_layers"] == 0:
            params["static_hidden"] = None
            params["static_dropout"] = None

        full_params = {**const_params, **params}
        param_sets.append(full_params)

    return param_sets


def sample_transformer_param_sets(const_params,
                                  param_grid,
                                  n_samples,
                                  seed=42):
    all_params = generate_transformer_param_sets(const_params, param_grid)
    print(f"Total valid combinations: {len(all_params)}")
    if n_samples >= len(all_params):
        print(f"Requested {n_samples}, using all {len(all_params)}.")
        return all_params
    rng = random.Random(seed)
    return rng.sample(all_params, n_samples)

In [ ]:
# ── GS setup — reuses generate_transformer_param_sets from 2.0 ────────────────
import csv, hashlib, json, random
from itertools import product
from datetime import datetime

GS_RESULTS_DIR = Path(cfg.dataPath) / path_cache / "FD_MODEL/GS_results"
GS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_GS = False

const_params = {
    "Fm": assets_full["ds_train"].Xm.shape[-1],  # 8
    "Fs": assets_full["ds_train"].Xs.shape[-1],  # 3
    "two_heads": False,
    "loss_spec": None,
    "T_max": 32,
}

param_grid = {
    "d_model":         [64, 128, 256],
    "nhead":           [2, 4, 8],          # filtered: must divide d_model
    "num_layers":      [2, 3, 4],
    "dim_feedforward": [128, 256, 512],    # filtered: must be >= d_model
    "dropout":         [0.0, 0.1, 0.2],
    "static_layers":   [1, 2],
    "static_hidden":   [64, 128],
    "static_dropout":  [0.0, 0.1],
    "head_dropout":    [0.0, 0.05, 0.1],
    "lr":              [5e-4, 1e-3, 2e-3],
    "weight_decay":    [1e-5, 1e-4],
}

N_SAMPLES = 150
sampled_params = sample_transformer_param_sets(
    const_params, param_grid, n_samples=N_SAMPLES, seed=cfg.seed,
)
print(f"Running {len(sampled_params)} random configurations")

In [ ]:
# ── Build GS train and val datasets from monthly data ─────────────────────────

# Collect all val glaciers per region from 2.3 within-region splits
WITHIN_SPLITS_CACHE = Path(cfg.dataPath) / path_cache / "WITHIN_REGION" / "within_region_splits_cache.pkl"
with open(WITHIN_SPLITS_CACHE, "rb") as f:
    _within_cache = pickle.load(f)
res_within_by_code = _within_cache["res_within_by_code"]

val_glaciers_by_region = {
    region: set(res_within_by_code[region]["test_glaciers"])
    for region in SOURCE_REGIONS
}
all_val_glaciers = set().union(*val_glaciers_by_region.values())

# Split all source glaciers into GS train / val lists
df_all_glaciers = pd.concat(
    [res_xreg_by_source[r]["df_train"][["GLACIER"]].drop_duplicates().assign(region=r)
     for r in SOURCE_REGIONS],
    ignore_index=True,
).rename(columns={"GLACIER": "glacier"})

gs_train_glaciers = df_all_glaciers[~df_all_glaciers["glacier"].isin(all_val_glaciers)]["glacier"].tolist()
gs_val_glaciers   = df_all_glaciers[ df_all_glaciers["glacier"].isin(all_val_glaciers)]["glacier"].tolist()

print(f"GS train glaciers : {len(gs_train_glaciers)}")
print(f"GS val glaciers   : {len(gs_val_glaciers)}")
for r, gls in val_glaciers_by_region.items():
    print(f"  {r} val: {len(gls)} glaciers")

In [ ]:
# Build datasets once — reused across all GS iterations
ds_gs_train_assets = build_assets_from_glacier_list(
    glaciers=gs_train_glaciers,
    df_ranked=df_all_glaciers,
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_path=str(CACHE_DIR / "assets_gs_train.joblib"),
    force_recompute=False,
    src_regions=SOURCE_REGIONS,
)

val_datasets = {}
for region in SOURCE_REGIONS:
    val_glaciers = list(val_glaciers_by_region[region])
    val_datasets[region] = build_assets_from_glacier_list(
        glaciers=val_glaciers,
        df_ranked=df_all_glaciers,
        res_xreg_by_source=res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        cache_path=str(CACHE_DIR / f"assets_gs_val_{region}.joblib"),
        force_recompute=False,
        src_regions=SOURCE_REGIONS,
    )

ds_gs_train = ds_gs_train_assets["ds_train"]
print(f"GS train sequences: {len(ds_gs_train)}")
for r, assets in val_datasets.items():
    print(f"  {r} val sequences: {len(assets['ds_train'])}")

In [ ]:
# All val glaciers combined (pooled val)
val_datasets["pooled"] = build_assets_from_glacier_list(
    glaciers=gs_val_glaciers,
    df_ranked=df_all_glaciers,
    res_xreg_by_source=res_xreg_by_source,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_path=str(CACHE_DIR / "assets_gs_val_pooled.joblib"),
    force_recompute=False,
    src_regions=SOURCE_REGIONS,
)

print(f"GS val pooled sequences: {len(val_datasets['pooled']['ds_train'])}")

In [ ]:
log_filename = GS_RESULTS_DIR / f"pooled_tf_gs_2026-06-12.csv"
if RUN_GS:    
    fieldnames = list(sampled_params[0].keys()) + \
                 ["valid_loss", "val_rmse_a", "val_rmse_w"] + \
                 [f"val_rmse_a_{r}" for r in SOURCE_REGIONS]

    with open(log_filename, mode="w", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

    best_overall = {"val": float("inf"), "row": None, "params": None}

    for i, params in enumerate(sampled_params):
        mbm.utils.seed_all(cfg.seed)
        run_id = hashlib.md5(json.dumps(params, sort_keys=True).encode()).hexdigest()[:8]
        print(f"\n--- Config {i+1}/{len(sampled_params)}  [id={run_id}] ---")

        # Fresh clone of train dataset — fit scalers here
        ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(ds_gs_train)
        ds_train_copy.fit_scalers(np.arange(len(ds_train_copy)))
        ds_train_copy.transform_inplace()
        train_dl = DataLoader(ds_train_copy, batch_size=64, shuffle=True)

        # Pooled val loader — apply train scalers
        ds_val_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            val_datasets["pooled"]["ds_train"]
        )
        val_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_val_copy, ds_train_copy, batch_size=128, seed=cfg.seed,
        )

        model = mbm.models.Transformer_MB.build_model_from_params(cfg, params, device, verbose=False)
        loss_fn = mbm.models.Transformer_MB.resolve_loss_fn(params)

        history, best_val, _ = model.train_loop(
            device=device,
            train_dl=train_dl,
            val_dl=val_dl,
            epochs=150,
            lr=params["lr"],
            weight_decay=params["weight_decay"],
            clip_val=1,
            sched_factor=0.5, sched_patience=6,
            sched_threshold=0.01, sched_threshold_mode="rel",
            sched_cooldown=1, sched_min_lr=1e-6,
            es_patience=15, es_min_delta=1e-4,
            log_every=10, verbose=False, loss_fn=loss_fn,
        )

        # Pooled val RMSE
        val_metrics, _ = model.evaluate_with_preds(device, val_dl, ds_val_copy)

        # Per-region val RMSE
        per_region_rmse = {}
        for region in SOURCE_REGIONS:
            ds_reg_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                val_datasets[region]["ds_train"]
            )
            reg_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_reg_copy, ds_train_copy, batch_size=128, seed=cfg.seed,
            )
            reg_metrics, _ = model.evaluate_with_preds(device, reg_dl, ds_reg_copy)
            per_region_rmse[f"val_rmse_a_{region}"] = reg_metrics["RMSE_annual"]

        print(f"  val_loss={best_val:.4f}  RMSE_a={val_metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={val_metrics['RMSE_winter']:.3f}  "
              + "  ".join(f"{r}={per_region_rmse[f'val_rmse_a_{r}']:.3f}" for r in SOURCE_REGIONS))

        row = {**params, "valid_loss": float(best_val),
               "val_rmse_a": float(val_metrics["RMSE_annual"]),
               "val_rmse_w": float(val_metrics["RMSE_winter"]),
               **per_region_rmse}
        with open(log_filename, mode="a", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

        if best_val < best_overall["val"]:
            best_overall = {"val": best_val, "row": row, "params": params}

    print("\n=== Best config ===")
    print(best_overall["params"])
    print(f"  val_loss={best_overall['val']:.4f}  val_rmse_a={best_overall['row']['val_rmse_a']:.3f}")

In [ ]:
results_gs = pd.read_csv(log_filename).sort_values(by='valid_loss')
results_gs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top10 = results_gs.head(10).reset_index(drop=True)

param_cols = [
    "d_model", "nhead", "num_layers", "dim_feedforward",
    "dropout", "static_layers", "static_hidden", "static_dropout",
    "head_dropout", "lr", "weight_decay",
]

n = len(param_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(param_cols):
    ax = axes[i]
    vals = top10[col].values
    colors = plt.cm.RdYlGn_r(top10["valid_loss"].rank(pct=True).values)
    bars = ax.barh(range(len(top10)), vals, color=colors)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels([f"#{j+1} ({row.valid_loss:.4f})" for j, row in top10.iterrows()], fontsize=8)
    ax.set_title(col, fontsize=10)
    ax.invert_yaxis()

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Top 10 GS configs by validation loss", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
best_row = results_gs.iloc[0]

PARAMS_TRANSFORMER_POOLED = {
    "Fm":             int(best_row["Fm"]),
    "Fs":             int(best_row["Fs"]),
    "d_model":        int(best_row["d_model"]),
    "nhead":          int(best_row["nhead"]),
    "num_layers":     int(best_row["num_layers"]),
    "dim_feedforward":int(best_row["dim_feedforward"]),
    "dropout":        float(best_row["dropout"]),
    "static_layers":  int(best_row["static_layers"]),
    "static_hidden":  None if pd.isna(best_row["static_hidden"]) else int(best_row["static_hidden"]),
    "static_dropout": None if pd.isna(best_row["static_dropout"]) else float(best_row["static_dropout"]),
    "head_dropout":   float(best_row["head_dropout"]),
    "lr":             float(best_row["lr"]),
    "weight_decay":   float(best_row["weight_decay"]),
    "two_heads":      False,
    "loss_spec":      None,
    "T_max":          32,
}

print("PARAMS_TRANSFORMER_POOLED = {")
for k, v in PARAMS_TRANSFORMER_POOLED.items():
    print(f'    "{k}": {repr(v)},')
print("}")

## Grid search DAN:

In [ ]:
# ── DAN sensitivity grid ───────────────────────────────────────────────────────
# Run after the transformer GS and model_foundation is trained
# Uses the same ft_assets_id splits as in 3.1

import itertools

DAN_GS_LOG = GS_RESULTS_DIR / f"dan_gs_{datetime.now().strftime('%Y-%m-%d')}.csv"
RUN_DAN_GS = True

dan_param_grid = {
    "dan_alpha":    [0.05, 0.1, 0.2],
    "mix_ratio_ft": [0.5, 1.0, 2.0],
}
# Fixed params
dan_fixed = {
    "grl_lambda":  1.0,
    "lr_backbone": 5e-5,
    "lr_domain":   1e-4,
    "epochs":      60,
}

dan_combos = [
    dict(zip(dan_param_grid.keys(), v))
    for v in itertools.product(*dan_param_grid.values())
]
print(f"DAN combinations: {len(dan_combos)}  ×  {len(TARGET_REGIONS_SUB)} regions "
      f"= {len(dan_combos) * len(TARGET_REGIONS_SUB)} runs")

In [ ]:
RECOMPUTE_SPLITS = False
FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_REGIONS = []

FT_FRAC = 0.10
save_path = BASE_DIR / "glacier_splits_FD.json"

gl_level_split = ['IT', 'CENTRALASIA', 'ALA']
id_level_split = [
    'IT', 'SJM', 'CENTRALASIA', 'CA_12', 'CA_3', 'CA_4', 'ALA', 'ALA_2',
    'ALA_4', 'ALA_6'
]

# ── Load existing glacier splits from disk ────────────────────────────────────
splits = {}
if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        splits = json.load(f)
    print(f"Loaded splits from: {save_path}")

# ── Only compute glacier splits for regions that need them ────────────────────
regions_needing_gl_split = [r for r in gl_level_split]
regions_to_compute = (regions_needing_gl_split if RECOMPUTE_SPLITS else
                      [r for r in regions_needing_gl_split if r not in splits])

for region in regions_to_compute:
    print(f"\n{'='*50}\nSplitting region: {region}")
    split = split_pool_holdout_sinkhorn(
        df_region=monthly_cache[region]["data_monthly"],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_joint=blur_joint,
        pool_frac=0.2,
        n_restarts=50,
        seed=cfg.seed,
    )
    splits[region] = split
    print(f"  Pool glaciers    : {split['pool_glaciers']}")
    print(f"  Holdout glaciers : {split['holdout_glaciers']}")
    print(f"  Pool fraction    : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout) : {split['sinkhorn_pool_vs_holdout']:.4f}")

if regions_to_compute:
    with open(save_path, "w") as f:
        json.dump(splits, f, indent=2)
    print(f"\nSaved splits for: {regions_to_compute}")

# ── Build FT assets for all target subregions ─────────────────────────────────
ft_assets_glacier = {}
ft_assets_id = {}

for region in TARGET_REGIONS_SUB:
    print(f"\n{'='*50}\nBuilding FT assets for: {region}")

    df_monthly = monthly_cache[region]["data_monthly"]
    df_monthly_aug = monthly_cache[region]["data_monthly_aug"]
    head_pad = monthly_cache[region]["months_head_pad"]
    tail_pad = monthly_cache[region]["months_tail_pad"]
    force = FORCE_RECOMPUTE or (region in FORCE_RECOMPUTE_REGIONS)

    # 1. Glacier-level split
    if region in gl_level_split:
        pool_glaciers = splits[region]["pool_glaciers"]
        holdout_glaciers = splits[region]["holdout_glaciers"]
        exp_key = f"FD_to_{region}"

        ds_ft_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_FT",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(pool_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                pool_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_gl = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key}_TEST",
            df_loss=df_monthly[df_monthly["GLACIER"].isin(holdout_glaciers)],
            df_full=df_monthly_aug[df_monthly_aug["GLACIER"].isin(
                holdout_glaciers)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )
        ft_train_idx_gl, ft_val_idx_gl = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_gl), val_ratio=0.2, seed=cfg.seed)
        ft_assets_glacier[region] = {
            "ds_ft": ds_ft_gl,
            "ds_test": ds_test_gl,
            "ft_train_idx": ft_train_idx_gl,
            "ft_val_idx": ft_val_idx_gl,
        }
        print(
            f"  [glacier-split] ds_ft  : {len(ds_ft_gl)} seqs | ds_test: {len(ds_test_gl)} seqs"
        )

    # 2. ID-level split
    if region in id_level_split:
        winter_ids = df_monthly[df_monthly["PERIOD"] ==
                                "winter"]["ID"].unique().tolist()
        annual_ids = df_monthly[df_monthly["PERIOD"] ==
                                "annual"]["ID"].unique().tolist()

        rng = np.random.default_rng(cfg.seed)
        rng.shuffle(winter_ids)
        rng.shuffle(annual_ids)

        n_ft_w = max(1, int(len(winter_ids) * FT_FRAC)) if winter_ids else 0
        n_ft_a = max(1, int(len(annual_ids) * FT_FRAC))
        ft_ids = winter_ids[:n_ft_w] + annual_ids[:n_ft_a]
        test_ids = winter_ids[n_ft_w:] + annual_ids[n_ft_a:]

        exp_key_id = f"FD_to_{region}_IDsplit"
        ds_ft_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_FT",
            df_loss=df_monthly[df_monthly["ID"].isin(ft_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(ft_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="ft",
            show_progress=False,
        )
        ds_test_id = build_or_load_lstm_dataset_only(
            cfg=cfg,
            key=f"{exp_key_id}_TEST",
            df_loss=df_monthly[df_monthly["ID"].isin(test_ids)],
            df_full=df_monthly_aug[df_monthly_aug["ID"].isin(test_ids)],
            months_head_pad=head_pad,
            months_tail_pad=tail_pad,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=str(CACHE_DIR),
            force_recompute=force,
            kind="test",
            show_progress=False,
        )
        ft_train_idx_id, ft_val_idx_id = mbm.data_processing.MBSequenceDataset.split_indices(
            len(ds_ft_id), val_ratio=0.2, seed=cfg.seed)
        ft_assets_id[region] = {
            "ds_ft": ds_ft_id,
            "ds_test": ds_test_id,
            "ft_train_idx": ft_train_idx_id,
            "ft_val_idx": ft_val_idx_id,
        }
        print(
            f"  [ID-split]      ds_ft  : {len(ds_ft_id)} seqs | ds_test: {len(ds_test_id)} seqs"
        )

# convenience dict: glacier-split where available, ID-split for the rest
ft_assets = {
    **ft_assets_glacier,
    **{
        r: ft_assets_id[r]
        for r in ft_assets_id if r not in ft_assets_glacier
    },
}

In [ ]:
TRAIN_FLAG = False
models_dir = BASE_DIR / "models" / "foundation"
os.makedirs(models_dir, exist_ok=True)

model_foundation, model_path, info = train_or_load_one_source_model(
    cfg=cfg,
    key="foundation_100pct",
    lstm_assets=assets_full,
    best_params=PARAMS_TRANSFORMER,
    device=device,
    models_dir=models_dir,
    prefix="transformer_foundation",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=N_EPOCHS,
    date=MODEL_DATE,
    model_type="transformer",
    verbose=True,
)

print(f"Pooled model saved to: {model_path}")

In [ ]:
# Build scaler once outside the loop
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    assets_full["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=assets_full["train_idx"],
    val_idx=assets_full["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

source_codes_pretrain = build_source_codes_for_dataset(
    assets_full["ds_train"],
    pd.concat(
        [res_xreg_by_source[r]["df_train"] for r in SOURCE_REGIONS],
        ignore_index=True,
    ),
    source_col="SOURCE_CODE",
)

In [ ]:
EXCLUDE_REGIONS = {'CENTRALASIA', 'ALA'}
tgt_labels = [
    t for t in TARGET_REGIONS_SUB if t not in EXCLUDE_REGIONS
]

if RUN_DAN_GS:
    dan_fieldnames = ["region", "dan_alpha", "mix_ratio_ft", "RMSE_annual", "RMSE_winter", "R2_annual"]
    with open(DAN_GS_LOG, mode="w", newline="") as f:
        csv.DictWriter(f, fieldnames=dan_fieldnames).writeheader()

    dan_gs_results = []

    for region in tgt_labels:
        print(f"\n{'='*50}  DAN GS → {region}")
        source_codes_ft = [region] * len(ft_assets_id[region]["ds_ft"])

        for combo in dan_combos:
            mbm.utils.seed_all(cfg.seed)
            params = {**dan_fixed, **combo}

            # temp model file — overwritten each combo, deleted after eval
            tmp_path = str(GS_RESULTS_DIR / f"_tmp_dan_{region}.pt")

            model_dan, _ = finetune_dan_transformer_on_target(
                cfg=cfg,
                model_foundation=model_foundation,
                assets_full=assets_full,
                ft_assets_region=ft_assets_id[region],
                ds_pooled_scaler=ds_pooled_scaler,
                source_codes_pretrain=source_codes_pretrain,
                source_codes_ft=source_codes_ft,
                device=device,
                best_params=PARAMS_TRANSFORMER_POOLED,
                model_filename=tmp_path,
                dan_alpha=params["dan_alpha"],
                grl_lambda=params["grl_lambda"],
                mix_ratio_ft=params["mix_ratio_ft"],
                epochs=params["epochs"],
                lr_backbone=params["lr_backbone"],
                lr_domain=params["lr_domain"],
                force_retrain=True,
                verbose=True,
            )

            # Evaluate on test set
            ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                ft_assets_id[region]["ds_test"]
            )
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_test_copy, ds_pooled_scaler, batch_size=128, seed=cfg.seed,
            )
            metrics, _ = model_dan.evaluate_with_preds(device, test_dl, ds_test_copy)

            if os.path.exists(tmp_path):
                os.remove(tmp_path)

            row = {"region": region, **combo,
                   "RMSE_annual": metrics["RMSE_annual"],
                   "RMSE_winter": metrics["RMSE_winter"],
                   "R2_annual":   metrics["R2_annual"]}
            dan_gs_results.append(row)

            with open(DAN_GS_LOG, mode="a", newline="") as f:
                csv.DictWriter(f, fieldnames=dan_fieldnames).writerow(row)

            print(f"  alpha={combo['dan_alpha']}  mix={combo['mix_ratio_ft']}  "
                  f"RMSE_a={metrics['RMSE_annual']:.3f}  RMSE_w={metrics['RMSE_winter']:.3f}")

    df_dan_gs = pd.DataFrame(dan_gs_results)
    print("\n=== Best DAN params per region ===")
    best_dan_by_region = df_dan_gs.loc[df_dan_gs.groupby("region")["RMSE_annual"].idxmin()]
    print(best_dan_by_region[["region", "dan_alpha", "mix_ratio_ft", "RMSE_annual", "RMSE_winter"]].to_string(index=False))

In [ ]:
best_dan_by_region

In [ ]:
# Best DAN params for the pooled model: minimize mean RMSE_annual across all regions
df_dan_gs_pooled = df_dan_gs.groupby(["dan_alpha", "mix_ratio_ft"])["RMSE_annual"].mean().reset_index()
best_dan_pooled = df_dan_gs_pooled.loc[df_dan_gs_pooled["RMSE_annual"].idxmin()]

PARAMS_DAN_POOLED = {
    **dan_fixed,
    "dan_alpha":    float(best_dan_pooled["dan_alpha"]),
    "mix_ratio_ft": float(best_dan_pooled["mix_ratio_ft"]),
}

print("PARAMS_DAN_POOLED =", PARAMS_DAN_POOLED)
print(f"  avg RMSE_annual across regions: {best_dan_pooled['RMSE_annual']:.4f}")
print()

# Optional: see the full ranking
print("All combos ranked by mean RMSE_annual:")
print(df_dan_gs_pooled.sort_values("RMSE_annual").to_string(index=False))